# Atomiq — Hybrid Quantum-Classical Term Deposit Campaign Pipeline

**The Atomiq Hackathon | Qiskit Fall Fest 2026 at IEM** — solo entry by Umar Shareef (UAF BSCS).

This notebook runs the full three-stage pipeline end to end and verifies every result:

1. **Predict** subscription probability for 100 test customers (hybrid classifier).
2. **Select** exactly 6 of the top 10 with QAOA (maximise expected net value).
3. **Simulate** P(at least 3 of the 6 subscribe) with a 6-qubit circuit.

**How to run:** create the venv, install `requirements.txt`, then Run All. Everything is seed-fixed (seed 42); the run takes about a minute and regenerates every result file in the repo root. Full technical report: `README.md`.

In [1]:
import json
from pathlib import Path
import pandas as pd

SEED = 42
ROOT = Path('.').resolve()
DATA = ROOT / 'resources' / 'bank_campaign_participant_release.xlsx'
print('repo root:', ROOT)
print('dataset exists:', DATA.exists())

repo root: /home/hatch/workspace/atomiq
dataset exists: True


In [2]:
# Run the whole pipeline (Stage 1 + Stage 2 + Stage 3), seed 42, ~1 minute.
# Regenerates stage1_predictions.csv, stage2_selection.csv/.json,
# stage3_simulation.csv/.json and results_summary.json in the repo root.
%run run_pipeline.py

train=300 test=100 train positives=35 (0.117)


CV logistic_regression: {'roc_auc': 0.6286, 'pr_auc': 0.2001, 'precision_at_10': 0.3, 'recall_at_10': 0.0857, 'brier': 0.2025, 'log_loss': 0.5955}


CV random_forest: {'roc_auc': 0.6467, 'pr_auc': 0.2331, 'precision_at_10': 0.3, 'recall_at_10': 0.0857, 'brier': 0.1011, 'log_loss': 0.3555}


CV grad_boosting: {'roc_auc': 0.5663, 'pr_auc': 0.1924, 'precision_at_10': 0.4, 'recall_at_10': 0.1143, 'brier': 0.114, 'log_loss': 0.524}
quantum-kernel features (train-only MI): [('pdays', 0.06), ('campaign', 0.0259), ('balance', 0.017), ('day', 0.0034)]


loaded cached quantum kernels
CV quantum_kernel_svc: {'roc_auc': 0.4905, 'pr_auc': 0.1204, 'precision_at_10': 0.1, 'recall_at_10': 0.0286, 'brier': 0.1034, 'log_loss': 0.3622}
CV hybrid_avg_logreg_qkernel: {'roc_auc': 0.6293, 'pr_auc': 0.1972, 'precision_at_10': 0.3, 'recall_at_10': 0.0857, 'brier': 0.1228, 'log_loss': 0.4093}
CV hybrid_avg_rf_qkernel: {'roc_auc': 0.6437, 'pr_auc': 0.2221, 'precision_at_10': 0.2, 'recall_at_10': 0.0571, 'brier': 0.1003, 'log_loss': 0.3474}
FINAL Stage 1 model: random_forest (best OOF PR-AUC, Brier tie-break)


Stage 1 top 10:
customer_id  predicted_probability  rank  is_top_10
      C0031               0.420000     1          1
      C0079               0.413333     2          1
      C0081               0.373333     3          1
      C0256               0.350000     4          1
      C0338               0.343333     5          1
      C0183               0.330000     6          1
      C0170               0.320000     7          1
      C0378               0.283333     8          1
      C0327               0.276667     9          1
      C0188               0.260000    10          1
Stage 2 candidates (net value = p*deposit - cost):
customer_id  predicted_probability  expected_deposit_value  contact_cost  capacity_units  expected_net_value
      C0031               0.420000                      13             2               1            3.460000
      C0079               0.413333                      24             3               3            6.919992
      C0081               0.373333

inequalities verified vacuous: max 6-subset cost=16.0 (<=25), max capacity=18.0 (<=22)


QAOA p=2: most-probable x=['C0031', 'C0079', 'C0256', 'C0338', 'C0170', 'C0327'] feasible=True Z=33.2167 (expectation -13.2320)


QAOA p=3: most-probable x=['C0079', 'C0081', 'C0256', 'C0183', 'C0378', 'C0188'] feasible=True Z=20.9333 (expectation -14.7886)
QAOA best feasible: ['C0031', 'C0079', 'C0256', 'C0338', 'C0170', 'C0327'] Z=33.2167 gap=0.0
Stage 3 probabilities: [0.42, 0.4133, 0.35, 0.3433, 0.32, 0.2767]
Stage 3 RY angles (rad): [1.4101, 1.3966, 1.2661, 1.2521, 1.2025, 1.1078]


  sensitivity: P(>=3) 0.3603 -> [0.3200, 0.4014] for +/-0.02 on all p; max single-customer shift 0.0069
Stage 3: quantum=0.3574 exact=0.3603 MC=0.3595 noisy=0.3787
  transpiled depth=2 gates={'ry': 6, 'measure': 6}

WROTE: stage1_predictions.csv, stage2_selection.csv/.json, stage3_simulation.csv/.json, results_summary.json


In [3]:
summary = json.loads((ROOT / 'results_summary.json').read_text())
print('Selected customers:', summary['stage2']['selected_ids'])
print('Expected net value (QAOA):', summary['stage2']['objective_qaoa'])
print('Brute-force optimum:    ', summary['stage2']['objective_brute_force'])
print('Optimality gap:         ', summary['stage2']['optimality_gap'])
print('Cost:     ', summary['stage2']['total_cost'], '/ 25')
print('Capacity: ', summary['stage2']['total_capacity'], '/ 22')
print('P(>=3 subscribe) quantum:', summary['stage3']['quantum_estimate_P_ge3'])
print('P(>=3 subscribe) exact:  ', summary['stage3']['exact_analytical_P_ge3'])

Selected customers: ['C0031', 'C0079', 'C0256', 'C0338', 'C0170', 'C0327']
Expected net value (QAOA): 33.2167
Brute-force optimum:     33.2167
Optimality gap:          0.0
Cost:      10.0 / 25
Capacity:  15.0 / 22
P(>=3 subscribe) quantum: 0.357422
P(>=3 subscribe) exact:   0.360265


## Stage 1 — prediction (hybrid classifier)

Preprocessing is fit on train only (no leakage). Six candidates are compared by 5-fold stratified cross-validation on train only; the final model is chosen by a pre-registered rule (best out-of-fold PR-AUC, Brier as tie-break). The quantum kernel classifier (4-qubit ZZ feature map) is built and reported honestly — it carries almost no signal on these features, so it is not the submitted model.

In [4]:
s1 = pd.read_csv(ROOT / 'stage1_predictions.csv')
print('test rows:', len(s1))
display(s1.head(10))
print()
print('CV metrics (out-of-fold, train only):')
for name, m in summary['stage1']['cv_metrics_oof_train_only'].items():
    print(f"  {name:28s} PR-AUC {m['pr_auc']:.4f}  P@10 {m['precision_at_10']:.2f}  R@10 {m['recall_at_10']:.4f}  Brier {m['brier']:.4f}")
print('Final model:', summary['stage1']['final_model'])

test rows: 100


,customer_id,predicted_probability,rank,is_top_10
0,C0031,0.420000,1,1
1,C0079,0.413333,2,1
2,C0081,0.373333,3,1
3,C0256,0.350000,4,1
4,C0338,0.343333,5,1
5,C0183,0.330000,6,1
6,C0170,0.320000,7,1
7,C0378,0.283333,8,1
8,C0327,0.276667,9,1
9,C0188,0.260000,10,1



CV metrics (out-of-fold, train only):
  logistic_regression          PR-AUC 0.2001  P@10 0.30  R@10 0.0857  Brier 0.2025
  random_forest                PR-AUC 0.2331  P@10 0.30  R@10 0.0857  Brier 0.1011
  grad_boosting                PR-AUC 0.1924  P@10 0.40  R@10 0.1143  Brier 0.1140
  quantum_kernel_svc           PR-AUC 0.1204  P@10 0.10  R@10 0.0286  Brier 0.1034
  hybrid_avg_logreg_qkernel    PR-AUC 0.1972  P@10 0.30  R@10 0.0857  Brier 0.1228
  hybrid_avg_rf_qkernel        PR-AUC 0.2221  P@10 0.20  R@10 0.0571  Brier 0.1003
Final model: random_forest


## Stage 2 — selection (QAOA vs brute force)

Maximise expected net value over the top 10, subject to exactly 6 selected, total contact cost ≤ 25 and total capacity ≤ 22. The QUBO enforces only the cardinality constraint (penalty P = 15); cost and capacity never bind (largest 6-subset cost 16, capacity 18) and are checked on every sampled bitstring. The brute-force check over all 210 combinations makes the optimum certain.

In [5]:
s2 = pd.read_csv(ROOT / 'stage2_selection.csv')
display(s2)
s2j = json.loads((ROOT / 'stage2_selection.json').read_text())
print('Selected:', s2j['selected_customer_ids'])
print('QAOA objective:', s2j['objective_value_qaoa'],
      '| Brute-force optimum:', s2j['objective_value_brute_force_optimum'],
      '| Gap:', s2j['optimality_gap'])
for run in s2j['qaoa_runs']:
    print(f"p={run['qaoa_reps']}: most-probable {run['most_probable_selected']} "
          f"(objective {run['most_probable_objective']})")

,customer_id,predicted_probability,expected_deposit_value,contact_cost,capacity_units,expected_net_value,selected
0,C0031,0.420000,13,2,1,3.4600,1
1,C0079,0.413333,24,3,3,6.9200,1
2,C0081,0.373333,17,4,1,2.3467,0
3,C0256,0.350000,22,1,4,6.7000,1
4,C0338,0.343333,18,2,2,4.1800,1
5,C0183,0.330000,14,2,3,2.6200,0
6,C0170,0.320000,22,1,1,6.0400,1
7,C0378,0.283333,14,3,1,0.9667,0
8,C0327,0.276667,25,1,4,5.9167,1
9,C0188,0.260000,13,2,2,1.3800,0


Selected: ['C0031', 'C0079', 'C0256', 'C0338', 'C0170', 'C0327']
QAOA objective: 33.2167 | Brute-force optimum: 33.2167 | Gap: 0.0
p=2: most-probable ['C0031', 'C0079', 'C0256', 'C0338', 'C0170', 'C0327'] (objective 33.2167)
p=3: most-probable ['C0079', 'C0081', 'C0256', 'C0183', 'C0378', 'C0188'] (objective 20.9333)


## Stage 3 — outcome simulation (6-qubit circuit)

One qubit per selected customer, `RY(2·arcsin(√p))` so P(|1⟩) = p. Success = at least 3 ones measured. Checked against the exact Poisson-binomial result and a 200k-trial Monte Carlo, plus a noisy-Aer run.

In [6]:
s3 = pd.read_csv(ROOT / 'stage3_simulation.csv')
display(s3)
s3j = json.loads((ROOT / 'stage3_simulation.json').read_text())
print('Qubit order:', s3j['qubit_order'])
print('Quantum P(>=3):   ', s3j['quantum_estimate_P_ge3'])
print('Exact P(>=3):     ', s3j['exact_analytical_P_ge3'])
print('Monte Carlo P(>=3):', s3j['monte_carlo_P_ge3_200k'])
print('Noisy P(>=3):     ', s3j['noisy_estimate_P_ge3'])
print('Error (quantum - exact):', s3j['error_quantum_minus_exact'])
print('Sensitivity: P(>=3) range for +/-0.02 on all p:',
      s3j['sensitivity_P_ge3_all_minus_0.02'], '-', s3j['sensitivity_P_ge3_all_plus_0.02'],
      '| max single-customer shift:', s3j['sensitivity_P_ge3_max_single_0.02'])
print('Transpiled depth:', s3j['transpiled_circuit_depth'],
      '| gates:', s3j['transpiled_gate_counts'], '| SWAPs:', s3j['swap_count'])

,customer_id,qubit,predicted_probability,ry_angle_rad
0,C0031,0,0.420000,1.410106
1,C0079,1,0.413333,1.396582
2,C0256,2,0.350000,1.266104
3,C0338,3,0.343333,1.252095
4,C0170,4,0.320000,1.202528
5,C0327,5,0.276667,1.107761


Qubit order: ['qubit 0 = C0031', 'qubit 1 = C0079', 'qubit 2 = C0256', 'qubit 3 = C0338', 'qubit 4 = C0170', 'qubit 5 = C0327']
Quantum P(>=3):    0.357422
Exact P(>=3):      0.360265
Monte Carlo P(>=3): 0.3595
Noisy P(>=3):      0.378723
Error (quantum - exact): -0.002843
Sensitivity: P(>=3) range for +/-0.02 on all p: 0.319971 - 0.401376 | max single-customer shift: 0.006873
Transpiled depth: 2 | gates: {'ry': 6, 'measure': 6} | SWAPs: 0


## Conclusion

Contact **C0031, C0079, C0256, C0338, C0170, C0327** — expected net value **33.22**, cost 10/25, capacity 15/22, P(at least 3 subscribe) ≈ **0.36**. Every quantum result is checked against an exact classical reference; no quantum-advantage claim is made. See `README.md` for the full technical report.